## Overview of Assignment 4

This assignment focuses on exploring and implementing advanced concepts and techniques in information retrieval. The primary objectives are to build Retrieval Augumentation Generation, and learn about Language Models

## Enter your details below

## Name

MD SANIM FARHAN

## Banner ID

B00906667

## GitHub Link of your Assingment 4

## Q1 : Setting up the libraries and the environment

In [2]:
# Q1(a) - Install required libraries

!pip install transformers
!pip install datasets
!pip install peft
!pip install accelerate
!pip install bitsandbytes
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install sentence-transformers
!pip install faiss-cpu

## Q2:  Data Preprocessing and Model Selection

In [3]:
import os
import tempfile

# Create a temporary directory for the dataset
docs_dir = tempfile.mkdtemp()

# Sample dataset about Artificial Intelligence and Machine Learning
documents = {
    "machine_learning.txt": """
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.
""",

    "deep_learning.txt": """
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.
""",

    "rag.txt": """
Retrieval-Augmented Generation, also called RAG, combines information
retrieval with large language models.

In a RAG system, documents are divided into smaller chunks and converted
into vector embeddings. When a user asks a question, the system retrieves
relevant chunks and provides them to the language model as context.

RAG can help language models answer questions using information stored
in external documents.
"""
}

# Save documents as text files
for filename, content in documents.items():
    file_path = os.path.join(docs_dir, filename)

    with open(file_path, "w", encoding="utf-8") as file:
        file.write(content)

print(f"Created {len(documents)} documents.")
print(f"Dataset directory: {docs_dir}")

Created 3 documents.
Dataset directory: C:\Users\sanim\AppData\Local\Temp\tmpy5y9zbaz


In [4]:
# we use the Textloader and load the documents

In [5]:
from langchain_community.document_loaders import TextLoader 
all_documents = []

for filename in documents.keys():
    file_path = os.path.join(docs_dir,filename)

    loader = TextLoader(file_path)
    loaded_docs = loader.load()

    all_documents.extend(loaded_docs)

print(f"Loaded {len(all_documents)} documets")

C:\Users\sanim\AppData\Local\Temp\ipykernel_14872\3588072547.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded 3 documets


In [6]:
# Split the text into chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter =RecursiveCharacterTextSplitter(
    chunk_size =500,
    chunk_overlap=50,
    separators = ["\n\n", "\n", ".", " ", ""]
)

document_chunks = text_splitter.split_documents(all_documents)
print(f"Created {len(document_chunks)} document chunks.")

print(f"\n Sample chunk:")
print(document_chunks[0].page_content)

print("\nMetadata: ")
print(document_chunks[0].metadata)

Created 3 document chunks.

 Sample chunk:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Metadata: 
{'source': 'C:\\Users\\sanim\\AppData\\Local\\Temp\\tmpy5y9zbaz\\machine_learning.txt'}


### Q2.1 Dataset Preprocessing

A small text dataset containing information about artificial intelligence,
machine learning, deep learning, and RAG is used. The documents are loaded
using TextLoader and divided into smaller chunks using
RecursiveCharacterTextSplitter. A chunk size of 500 and an overlap of 50
are used to preserve context between neighbouring chunks.

In [8]:
#loading tokenizer 

from transformers import AutoTokenizer 

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

C:\Users\sanim\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanim\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [9]:
# Tokenize the document chunks 
tokenized_chunks = []

for chunk in document_chunks:
    tokens = tokenizer(
        chunk.page_content,
        truncation=True,
        padding=False,
        max_length=512,
        return_tensors="pt"
    )

    tokenized_chunks.append(tokens)

print(f"Tokenized {len(tokenized_chunks)} chunks.")


Tokenized 3 chunks.


In [10]:
# Showing that tokenization worked 
print("Original text:")
print(document_chunks[0].page_content)

print("\nToken IDs:")
print(tokenized_chunks[0]["input_ids"])

print("\nNumber of tokens:")
print(tokenized_chunks[0]["input_ids"].shape[1])

Original text:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Token IDs:
tensor([[    1,  6189,  6509,   338,   263,  5443,   310, 23116, 21082,   393,
          6511, 23226,    13,   517,  5110, 15038,   515,   848,  1728,  1641,
          9479,  1824,  2168, 29889,    13,    13,  8439,   526,  2211,  3619,
          4072,   310,  4933,  6509, 29901,  2428, 11292,  6509, 29892,    13,
           348,  9136, 11292,  6509, 29892,   322, 15561,  1454, 13561,  6509,
         29889,    13,    13, 19111, 11292,  6509,  3913,  3858,   839,   848,
           304,  7945,  4733, 29889, 13103,  2428, 11292,    13, 21891,  9595,
          3160, 12965,   

### Q2.2 Tokenization

The TinyLlama tokenizer is used because TinyLlama is the pretrained
language model selected for the RAG system. The tokenizer converts each
text chunk into token IDs that can be understood by the language model.
The maximum sequence length is set to 512 tokens and truncation is enabled
to prevent sequences from exceeding this limit.

In [11]:
# Q2.3 - Split tokenized data into chunks for indexing

token_chunks = []

chunk_size = 256
chunk_overlap = 50

for tokens in tokenized_chunks:
    input_ids = tokens["input_ids"][0]

    start = 0

    while start < len(input_ids):
        end = start + chunk_size

        chunk = input_ids[start:end]

        token_chunks.append(chunk)

        if end >= len(input_ids):
            break

        start += chunk_size - chunk_overlap

print(f"Created {len(token_chunks)} token chunks.")

Created 3 token chunks.


### Q2.3 Token Chunking

The tokenized documents are divided into smaller chunks of 256 tokens with
an overlap of 50 tokens. The overlap helps preserve context between consecutive
chunks. These smaller chunks can later be converted into embeddings and stored
in a vector index for efficient retrieval.

In [12]:
# Convert token chunks back into text for embedding
indexing_chunks = []

for chunk in token_chunks:
    text = tokenizer.decode(chunk, skip_special_tokens=True)
    indexing_chunks.append(text)

print(f"Prepared {len(indexing_chunks)} chunks for indexing.")
print("\nSample chunk:")
print(indexing_chunks[0])

Prepared 3 chunks for indexing.

Sample chunk:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.


In [14]:
# Create embeddings
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(indexing_chunks)

print("Embedding shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (3, 384)


In [15]:
import faiss
import numpy as np

# Convert embeddings to float32, which FAISS expects
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

# Get embedding dimension
dimension = chunk_embeddings.shape[1]

# Create FAISS L2 index
index = faiss.IndexFlatL2(dimension)

# Add embeddings to the index
index.add(chunk_embeddings)

print(f"FAISS index created successfully.")
print(f"Number of vectors stored: {index.ntotal}")

FAISS index created successfully.
Number of vectors stored: 3


In [16]:
query = "What is machine learning?"

query_embedding = embedding_model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

distances, indices = index.search(query_embedding, k=2)

print("Query:", query)

for i, idx in enumerate(indices[0]):
    print(f"\nResult {i + 1}:")
    print(indexing_chunks[idx])
    print("Distance:", distances[0][i])

Query: What is machine learning?

Result 1:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.
Distance: 0.43365866

Result 2:
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.
Distance: 0.9418586


### Q2.4 Vector Store

The `all-MiniLM-L6-v2` sentence-transformer model is used to convert the text chunks into numerical embedding vectors. These embeddings are stored in a FAISS `IndexFlatL2` vector index. FAISS allows relevant chunks to be efficiently retrieved by comparing the embedding of a user's query with the stored document embeddings using L2 distance.


## Q3: Implementing RAG using LangChain for different queries

### Q3.1 RAG Pipeline

Retrieval-Augmented Generation (RAG) combines information retrieval with a language model so that the model can answer questions using relevant external information.

The RAG pipeline used in this assignment contains the following main components:

1. **Documents/Data Source:**
   The original text documents contain the information that the system will use to answer questions.

2. **Text Chunking:**
   Large documents are divided into smaller chunks so that relevant sections can be retrieved more efficiently.

3. **Embedding Model:**
   The `all-MiniLM-L6-v2` sentence-transformer model converts each text chunk into a numerical embedding vector that represents its meaning.

4. **Vector Store:**
   FAISS stores the embedding vectors and allows fast similarity searching.

5. **User Query:**
   When a user asks a question, the query is also converted into an embedding using the same embedding model.

6. **Retriever:**
   FAISS compares the query embedding with the stored document embeddings and retrieves the most relevant chunks.

7. **Prompt Construction:**
   The retrieved chunks are added to the user's question as context for the language model.

8. **Language Model:**
   TinyLlama receives the question together with the retrieved context and generates the final answer.

Therefore, the overall RAG pipeline is:

**Documents → Chunking → Embeddings → FAISS Vector Store → User Query → Query Embedding → Relevant Chunk Retrieval → Prompt with Context → TinyLlama → Generated Answer**


### Q3.2 Pretrained Language Model Selection

For this RAG system, I selected **TinyLlama-1.1B-Chat-v1.0** as the pretrained language model. TinyLlama is a lightweight language model with approximately 1.1 billion parameters, making it suitable for running in environments with limited computational resources. The chat version is designed for instruction and conversational tasks, which makes it appropriate for answering user queries using the context retrieved by the RAG system. Its smaller size also provides faster inference compared with larger language models while still being suitable for demonstrating the RAG pipeline.


In [18]:
!pip install -U langchain-huggingface

In [19]:
# Creating a LangChain FAISS vector store
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_texts(
    texts=indexing_chunks,
    embedding=embeddings
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

print("LangChain FAISS vector store and retriever created.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LangChain FAISS vector store and retriever created.


In [20]:
# loading the actual TinyLlama model.
from transformers import AutoModelForCausalLM, pipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    do_sample=False
)

print("TinyLlama loaded successfully.")

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


TinyLlama loaded successfully.


In [22]:
# connecting the Hugging Face text-generation pipeline to LangChain.
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline
)

print("TinyLlama connected to LangChain successfully.")

TinyLlama connected to LangChain successfully.


In [23]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:"""
)

print("Prompt template created successfully.")

Prompt template created successfully.


In [24]:
# building the actual RAG chain.
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


In [25]:
# testing rag chain 
query = "What is machine learning?"

response = rag_chain.invoke(query)

print("Question:")
print(query)

print("\nRAG Response:")
print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question:
What is machine learning?

RAG Response:
Answer the question using only the provided context.

Context:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.

Question:
What is machine learning?

Answer:Machine learning is a branch of artificial intelligence that allows
computers to learn patterns from data without being explicitly programmed.


### Q3.3 RAG Pipeline Implementation

The RAG pipeline was implemented using LangChain. The `all-MiniLM-L6-v2` embedding model is used with FAISS to store and retrieve relevant document chunks. FAISS is used as a retriever to find the most relevant chunks for a user's query. TinyLlama-1.1B-Chat-v1.0 is connected to LangChain using a Hugging Face text-generation pipeline. The retrieved chunks are then added to the prompt as context along with the user's question. Finally, TinyLlama uses this context to generate the answer.


In [26]:
# Q3.4 - Test the RAG pipeline with meaningful queries

queries = [
    "What are the three common types of machine learning?",
    "Where is deep learning commonly used?",
    "How does a RAG system work?"
]

for query in queries:
    response = rag_chain.invoke(query)

    print("=" * 60)
    print("Question:")
    print(query)

    print("\nRAG Response:")
    print(response)

    print()

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
What are the three common types of machine learning?

RAG Response:
Answer the question using only the provided context.

Context:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.

Question:
What are the three common types of machine learning?

Answer:
1. Supervised learning: This type of machine learning involves labelled data
to train models. Examples incl

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Where is deep learning commonly used?

RAG Response:
Answer the question using only the provided context.

Context:
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.

Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Question:
Where is deep learning commonly used?

Answer:
Deep learning is commonly used in computer vision, speech recognition,
and natural language processing.

Question:
How does a RAG system 

### Q3.5 Analysis of RAG Results

The generated responses show that the RAG model is able to retrieve relevant information from the dataset and use it to answer the queries. For example, when asked about the three common types of machine learning, the system correctly retrieved the machine learning document and identified supervised learning, unsupervised learning, and reinforcement learning.

The response was highly relevant to the query because the main answer matched the retrieved context. However, the model also generated some additional examples that were not included in the retrieved documents. This shows that the RAG pipeline improves the relevance of the response by providing useful context, but the language model may still generate some information from its pretrained knowledge.


## Q4 : Modify and evaluate the different components of RAG

In [27]:
# Q4.1 - Compare different retrieval techniques

# Normal similarity-based retriever
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# Maximal Marginal Relevance (MMR) retriever
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 2,
        "fetch_k": 3,
        "lambda_mult": 0.5
    }
)

print("Similarity and MMR retrievers created successfully.")

Similarity and MMR retrievers created successfully.


In [28]:
query = "How does machine learning relate to deep learning?"

similarity_docs = similarity_retriever.invoke(query)
mmr_docs = mmr_retriever.invoke(query)

print("=== Similarity Retrieval ===")
for i, doc in enumerate(similarity_docs, 1):
    print(f"\nDocument {i}:")
    print(doc.page_content)

print("\n" + "=" * 60)

print("=== MMR Retrieval ===")
for i, doc in enumerate(mmr_docs, 1):
    print(f"\nDocument {i}:")
    print(doc.page_content)

=== Similarity Retrieval ===

Document 1:
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.

Document 2:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

=== MMR Retrieval ===

Document 1:
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn

In [29]:
similarity_rag_chain = (
    {
        "context": similarity_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

mmr_rag_chain = (
    {
        "context": mmr_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Both RAG chains created successfully.")

Both RAG chains created successfully.


In [30]:
# running the same query through both RAG chains so we can compare the generated responses fairly.
query = "How does machine learning relate to deep learning?"

similarity_response = similarity_rag_chain.invoke(query)
mmr_response = mmr_rag_chain.invoke(query)

print("=== Similarity RAG Response ===")
print(similarity_response)

print("\n" + "=" * 60)

print("=== MMR RAG Response ===")
print(mmr_response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Similarity RAG Response ===
Answer the question using only the provided context.

Context:
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.

Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Question:
How does machine learning relate to deep learning?

Answer:
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers. Machine learning is a branch of artificial
intelli

### Q4.1 Comparison of Retrieval Techniques

Two retrieval techniques were tested: standard similarity search and Maximal Marginal Relevance (MMR). Both methods used the same query, language model, and prompt so that only the retrieval technique was changed.

For the query **“How does machine learning relate to deep learning?”**, similarity search retrieved the machine learning and deep learning chunks. These chunks were both directly related to the query, so the generated response clearly explained that deep learning is a subset of machine learning.

MMR retrieved the deep learning chunk and a RAG-related chunk. MMR tries to increase diversity among the retrieved documents, but in this case the additional RAG chunk was not relevant to the query. Because of this, the generated response included unnecessary information about RAG and was less focused.

Therefore, for this small dataset and this query, standard similarity retrieval produced a more relevant and focused response than MMR. However, MMR can be useful when the vector store contains many similar or repetitive documents because it attempts to return a more diverse set of relevant results.


In [32]:
# Create an improved prompt
from langchain_core.prompts import PromptTemplate

improved_prompt = PromptTemplate.from_template(
    """You are a question-answering assistant.

Use ONLY the information provided in the context below to answer the question.
Do not add information that is not present in the context.
Give a clear, concise, and direct answer.
If the answer cannot be found in the context, say:
"I cannot answer this question based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

print("Improved prompt created successfully.")

Improved prompt created successfully.


In [33]:
improved_rag_chain = (
    {
        "context": similarity_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | improved_prompt
    | llm
    | StrOutputParser()
)

print("Improved RAG chain created successfully.")

Improved RAG chain created successfully.


In [34]:
query = "What are the three common types of machine learning?"

original_response = similarity_rag_chain.invoke(query)
improved_response = improved_rag_chain.invoke(query)

print("=== Original Prompt Response ===")
print(original_response)

print("\n" + "=" * 60)

print("=== Improved Prompt Response ===")
print(improved_response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Original Prompt Response ===
Answer the question using only the provided context.

Context:
Machine learning is a branch of artificial intelligence that allows computers
to learn patterns from data without being explicitly programmed.

There are three common types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train models. Common supervised
learning tasks include classification and regression.

Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.

Question:
What are the three common types of machine learning?

Answer:
1. Supervised learning: This type of machine learning involves labelled data
to train models. Examples include classification and regression.

2. Unsupe

### Q4.2 Prompt Modification Analysis

The original prompt only instructed the model to answer using the provided context. I modified the prompt by adding stronger instructions to use only the retrieved context, avoid adding outside information, provide a concise answer, and state when the answer is not available in the context.

The modified prompt produced a more structured response, but it still added some information that was not present in the retrieved documents, such as examples of clustering and video games. Therefore, the prompt modification improved the guidance given to the model, but it did not completely prevent TinyLlama from using its pretrained knowledge.

This shows that prompt engineering can improve the behaviour of the RAG system, but the quality of the final response also depends on the language model itself.


In [35]:
# Q4.3 - Change the number of retrieved documents

retriever_k1 = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

retriever_k3 = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retrievers with k=1 and k=3 created successfully.")

Retrievers with k=1 and k=3 created successfully.


In [36]:
rag_chain_k1 = (
    {
        "context": retriever_k1 | format_docs,
        "question": RunnablePassthrough()
    }
    | improved_prompt
    | llm
    | StrOutputParser()
)

rag_chain_k3 = (
    {
        "context": retriever_k3 | format_docs,
        "question": RunnablePassthrough()
    }
    | improved_prompt
    | llm
    | StrOutputParser()
)

print("Both RAG chains created successfully.")

Both RAG chains created successfully.


In [37]:
query = "How does machine learning relate to deep learning?"

response_k1 = rag_chain_k1.invoke(query)
response_k3 = rag_chain_k3.invoke(query)

print("=== k = 1 Response ===")
print(response_k1)

print("\n" + "=" * 60)

print("=== k = 3 Response ===")
print(response_k3)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== k = 1 Response ===
You are a question-answering assistant.

Use ONLY the information provided in the context below to answer the question.
Do not add information that is not present in the context.
Give a clear, concise, and direct answer.
If the answer cannot be found in the context, say:
"I cannot answer this question based on the provided context."

Context:
Deep learning is a subset of machine learning that uses artificial neural
networks with multiple layers.

Deep learning is commonly used in computer vision, speech recognition,
and natural language processing. Neural networks learn useful representations
from large amounts of data.

Question:
How does machine learning relate to deep learning?

Answer:
Deep learning is a subfield of machine learning that uses neural networks
with multiple layers. Machine learning is a subset of artificial intelligence
that uses algorithms to learn from data.

Both deep learning and machine learning are used in computer vision, speech
recognit

### Q4.3 Effect of Number of Retrieved Documents

I compared the RAG pipeline using `k=1` and `k=3`. With `k=1`, only the deep learning document was retrieved. The response was focused on deep learning, but it added several details that were not available in the retrieved context.

With `k=3`, the system retrieved the deep learning, machine learning, and RAG documents. The response explained the relationship between machine learning and deep learning more clearly because it had information from both relevant documents. However, retrieving three documents also introduced the unrelated RAG document into the context.

Therefore, increasing the number of retrieved documents can provide more useful context and improve the answer, but retrieving too many documents may also introduce irrelevant information. A suitable value of `k` should balance sufficient context with relevance.


### Q4.4 Comparative Analysis of Original and Modified RAG Pipelines

The original RAG pipeline used similarity-based retrieval with `k=2` and a simple prompt that instructed TinyLlama to answer using the retrieved context. Several modifications were tested to examine how retrieval strategy, prompt design, and the number of retrieved documents affected the generated responses.

| Pipeline            | Modification                                        | Result                                                                                                                          |
| ------------------- | --------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------- |
| Original RAG        | Similarity retrieval, `k=2`, basic prompt           | Retrieved relevant documents and generated generally correct answers, but sometimes added information not found in the context. |
| MMR RAG             | MMR retrieval instead of similarity                 | Increased diversity of retrieved documents, but sometimes selected less relevant information.                                   |
| Improved Prompt RAG | Added stronger instructions to use only the context | Provided clearer guidance to the model, but TinyLlama still generated some information from its pretrained knowledge.           |
| RAG with `k=1`      | Retrieved only one document                         | Produced a more focused context, but sometimes lacked enough information to fully answer the question.                          |
| RAG with `k=3`      | Retrieved three documents                           | Provided more information, but also introduced unrelated context into the prompt.                                               |

A qualitative example was the query **“How does machine learning relate to deep learning?”** With similarity retrieval and `k=2`, the system retrieved both the machine learning and deep learning documents. This allowed TinyLlama to correctly explain that deep learning is a subset of machine learning. With MMR retrieval, the system retrieved the deep learning document together with an unrelated RAG document. As a result, the generated answer also started discussing RAG, making it less focused.

Changing the number of retrieved documents also affected the response. With `k=1`, the system only received information about deep learning and generated several additional details that were not available in the context. With `k=3`, the model received both the machine learning and deep learning information, which improved the main answer, but the unrelated RAG document was also included.

Overall, the experiments show that more retrieved information does not always produce a better response. For this small dataset, **similarity retrieval with `k=2` provided the best balance**, because it retrieved enough relevant information without introducing unnecessary documents. The improved prompt provided additional guidance, although TinyLlama still occasionally used information from its pretrained knowledge.


# Q5. Selecting and implementing a pretrained model for a new task (8 marks)

### Q5.1 New Task Selection

For this task, I selected **sentiment classification**. Sentiment classification is a text classification task where the model analyzes a piece of text and predicts the sentiment expressed in it, such as positive or negative.

This task is different from the previous RAG and question-answering tasks because the goal is not to retrieve information or generate an answer from context. Instead, the model directly classifies the input text into a predefined sentiment category.

For example:

**Input:** “The movie was very interesting and enjoyable.”

**Output:** Positive

**Input:** “The service was terrible and disappointing.”

**Output:** Negative


### Q5.2 Pretrained Model Selection

For the sentiment classification task, I selected **`distilbert-base-uncased-finetuned-sst-2-english`** from Hugging Face.

This model is based on the DistilBERT architecture and was fine-tuned on the labelled **SST-2 sentiment classification dataset**. Therefore, it uses supervised fine-tuning for the downstream classification task. The model predicts whether an English text expresses a **positive or negative sentiment**.

This model is suitable for the selected task because it has already been trained specifically for sentiment classification. It is also smaller and more computationally efficient than the original BERT model, making it practical to run in a Jupyter notebook.

This model is different from TinyLlama, which was used for the RAG pipeline in the previous questions.


In [39]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

print("Sentiment classification model loaded successfully.")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\sanim\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanim\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentiment classification model loaded successfully.


In [40]:
texts = [
    "The movie was amazing and I really enjoyed it.",
    "The service was terrible and very disappointing.",
    "I am very happy with this product."
]

results = sentiment_model(texts)

for text, result in zip(texts, results):
    print("Text:", text)
    print("Prediction:", result["label"])
    print("Confidence:", round(result["score"], 4))
    print("-" * 50)

Text: The movie was amazing and I really enjoyed it.
Prediction: POSITIVE
Confidence: 0.9999
--------------------------------------------------
Text: The service was terrible and very disappointing.
Prediction: NEGATIVE
Confidence: 0.9998
--------------------------------------------------
Text: I am very happy with this product.
Prediction: POSITIVE
Confidence: 0.9999
--------------------------------------------------


### Q5.3 Sentiment Classification Implementation

The sentiment classification task was implemented using the Hugging Face Transformers library. The pretrained `distilbert-base-uncased-finetuned-sst-2-english` model was loaded using the sentiment-analysis pipeline. Multiple input sentences were passed to the model, and the model predicted whether each sentence expressed positive or negative sentiment. The model also returned a confidence score for each prediction.
